# **Prova Estrazione Highlights**

In questa prova utilizzo il dataset Learning From The Worst la cui definizione usata è:

"‘Hate’ is defined as “abusive speech targeting specific group characteristics, such as ethnic origin, religion, gender, or sexual orientation."

Le componenti sono Abuse, Ethnicity, Gender, Religion, Sexual Orientation.

Questa prova è con Mistral-7b-Instruct-v0.3

# HF login e imports

In [1]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [ ]:
!hf auth login --token hf_**************

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `NLP2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `NLP2`


In [3]:
!pip install --upgrade transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import numpy as np
import json
import random
from IPython.display import display, HTML

# Data Processing

In [6]:
!gdown 1ygTn8Q6xVk_RNTae-0W-SigKdjHOkUTx

Downloading...
From: https://drive.google.com/uc?id=1ygTn8Q6xVk_RNTae-0W-SigKdjHOkUTx
To: /content/sampled_rows_LFTW.csv
100% 18.4k/18.4k [00:00<00:00, 36.1MB/s]


In [7]:
df=pd.read_csv('/content/sampled_rows_LFTW.csv')
df.head()

,text,type,target
0,"Burning TikTok isn't a bad thing, especially i...",animosity,asi.east
1,"Donald Trump, please make gay bullying accepta...",animosity,gay
2,In particular what I would say because of wher...,animosity,mixed.race
3,Yeah I have a fugly yt friend who had to resor...,animosity,bla
4,Muslim women are just big letterboxes,animosity,mus.wom


In [8]:
texts=df['text']
texts

,text
0,"Burning TikTok isn't a bad thing, especially i..."
1,"Donald Trump, please make gay bullying accepta..."
2,In particular what I would say because of wher...
3,Yeah I have a fugly yt friend who had to resor...
4,Muslim women are just big letterboxes
...,...
95,"yesterday, more people died of stabings than c..."
96,I believe genderbenders are gods error. they w...
97,"Nigeria has a population of 200mil, I say we t..."
98,bl ac ks will be physically gone from this pla...


# Load Model

In [9]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name="mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config
  )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Prompt

In [10]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse:  Content which involves improper treatment of a person or group, often to unfairly or improperly gain benefit.
        - Derogation: Content which explicitly attacks, demonizes, demeans or insults a group
        - Animosity: Content which expresses abuse against a group in an implicit or subtle manner.
        - Threatening language: Content which expresses intention to, support for, or encourages inflicting harm on a group, or identified members of the group.
        - Support for hateful entities: Content which explicitly glorifies, justifies or supports hateful actions, events, organizations, tropes and individuals.
        - Dehumanization Content which ‘perceiv[es] or treat[s] people as less than human’
        - Ethnicity: Any reference to a person's or group's ethnic origin.
        - Gender: Any reference to a person's or group's gender identity or biological sex.
        - Religion: Any reference to a person's or group's religious beliefs, practices, or affiliation.
        - Sexual Orientation: Any reference to a person's or group's sexual orientation.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Derogation": "extracted substring or absent",
            "Animosity": "extracted substring or absent",
            "Threatening language": "extracted substring or absent",
            "Support for hateful entities": "extracted substring or absent",
            "Dehumanization": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent"
        }

        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be ONLY the raw json object. Do not wrap the json in markdown formatting.

        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [11]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  formatted_prompts=[]

  for text in texts:
    prompt_copy = [
            {"role": prompt_template[0]["role"], "content": prompt_template[0]["content"]},
            {
                "role": prompt_template[1]["role"],
                "content": prompt_template[1]["content"].replace("{text}", text)
            }
        ]


    formatted_prompt = tokenizer.apply_chat_template(
      prompt_copy,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    )

    formatted_prompts.append(formatted_prompt)


  return formatted_prompts

# Inference

In [12]:
def generate_responses(model, prompt_examples, tokenizer, verbose=False):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """
  outputs=[]
  for index, prompt_example in enumerate(prompt_examples):
    if verbose and index % 10 == 0:
      print(f"generating response n.{index}...")
    prompt_example=prompt_example.to(model.device)
    output = model.generate(**prompt_example, max_new_tokens=500,pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0][prompt_example["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(output)

  return outputs

In [ ]:
outputs = generate_responses(model, prepare_prompts(texts, prompt, tokenizer), tokenizer, verbose=True)
#32

generating response n.0...
generating response n.10...
generating response n.20...
generating response n.30...
generating response n.40...
generating response n.50...
generating response n.60...
generating response n.70...
generating response n.80...
generating response n.90...


In [14]:
parsed_outputs = []
json_errors = []
for output in outputs:
    try:
        parsed_outputs.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors.append(output)

with open("results_LFTW_3.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs, f, ensure_ascii=False, indent=2)

with open("json_errors_LFTW_3.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors))
    f.write("\n")
    f.write("--"*20)

# Extraction Results

In [ ]:
texts = texts.reset_index(drop=True)

In [ ]:
for i in range(len(outputs)):
  print(texts[i])
  print(outputs[i])
  print('-'*20)

In [ ]:
def highlights(texts, outputs):

  """
      This function displays the given text with the model extracted highlights.

  """

  colori = {
        "Abuse": "#FFB3B3",
        "Ethnicity": "#B3D9FF",
        "Gender": "#E6B3FF",
        "Religion": "#B3FFB3",
        "Sexual Orientation": "#FFD9B3"
      }

  testo_colorato = texts

  for categoria, estratto in outputs.items():
      if estratto and estratto.lower() != "null" and estratto in testo_colorato:
          colore = colori.get(categoria, "#DDDDDD")
          tag_html = f"<mark style='background-color: {colore}; padding: 2px 4px; border-radius: 4px;'>{estratto} <span style='font-size: 0.7em; font-weight: bold;'>[{categoria}]</span></mark>"
          testo_colorato = testo_colorato.replace(estratto, tag_html)

  display(HTML(f"<p style='line-height: 1.8; font-size: 16px;'>{testo_colorato}</p>"))


In [ ]:
for i in range(len(outputs)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs[i]}")

    highlights(texts[i], parsed_output)

JSONDecodeError: Expecting ':' delimiter: line 2 column 58 (char 59) for text: Julie was smoking, like at least get up off the filthy ground, so fucking disgusting to smoke next to a child, dirty fucking trampJ, model output: {
  "Abuse": "so fucking disgusting", "dirty fucking tramp", "fucking disgusting"
}
